# Battery fleet investigator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThomasAko/battery-fit-agent/blob/main/notebooks/colab_investigator.ipynb)

This notebook runs the **investigator mode** of `battery-fit-agent` on all 139 Severson fast-charge cells, and keeps
the full record of what the agent did:

1. **Fit** seven degradation laws to every cell (deterministic tools, no LLM).
2. **Triage** each cell against the fleet and build a shortlist.
3. **Investigate** each shortlisted cell with a tool-using LLM agent. The agent chooses which tools to call and
   when it knows enough, then gives a structured verdict. A deterministic verifier checks the verdict against the
   tool results and sends it back once if they disagree.
4. **Report** from the verdicts, and score the verdicts against hand labels and a rule-based baseline.

Runs in Google Colab or in Jupyter from the repository. Works with Groq (free tier), Gemini, Claude, OpenRouter or a
local Ollama model; set the provider below. In Colab, store the API key as a secret (key icon in the left sidebar)
named as in the table below.

| Provider | Secret / environment variable | Notes |
|---|---|---|
| `groq` | `GROQ_API_KEY` | free tier at console.groq.com; GPT OSS 120B by default |
| `gemini` | `GOOGLE_API_KEY` | aistudio.google.com; also reads the plots |
| `anthropic` | `ANTHROPIC_API_KEY` | platform.claude.com; also reads the plots |
| `openrouter` | `OPENROUTER_API_KEY` | needs a model id |
| `ollama` | none | local only, not in Colab |

## Setup

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q https://github.com/ThomasAko/battery-fit-agent.git
    %cd battery-fit-agent
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..
print("working directory:", os.getcwd())

In [ ]:
import os
PROVIDER = "groq"      # groq | gemini | anthropic | openrouter | ollama
MODEL    = None        # None = provider default (see battery_fit_agent/llm.py)
RPM      = 2           # max requests per minute: ~2 for Groq's free tier, 8 for Gemini free, 20+ for paid APIs
WHICH    = "labelled"  # "labelled" (the 10 hand-labelled cells), "high" (high-priority shortlist) or "all"

from battery_fit_agent.llm import _KEYS
key_name = _KEYS.get(PROVIDER)
if key_name and not os.getenv(key_name):
    if IN_COLAB:
        from google.colab import userdata
        os.environ[key_name] = userdata.get(key_name)
    else:
        from getpass import getpass
        os.environ[key_name] = getpass(f"{key_name}: ")
print("provider:", PROVIDER, "| key set:", bool(key_name is None or os.getenv(key_name)))

## The agent

The outer workflow (fit, triage, investigate each cell, report) and, inside `investigate`, the investigator agent:
a loop between the model and its tools, then a structured verdict and a verifier.

In [ ]:
import os
import pandas as pd
from IPython.display import Markdown, Image, display
from battery_fit_agent import Config
from battery_fit_agent.llm import get_chat_model, supports_images
from battery_fit_agent.graph_v2 import build_fleet_agent
from battery_fit_agent.investigator import build_investigator

llm = get_chat_model(PROVIDER, MODEL, requests_per_minute=RPM or None)
print("model:", getattr(llm, "model", None) or getattr(llm, "model_name", None))

cfg = Config(out_dir="agent_outputs_investigator"); cfg.make_dirs()
labels = pd.read_csv("results/manual_labels.csv")
agent = build_fleet_agent(cfg, llm, which="cells" if WHICH == "labelled" else WHICH,
                          cells=labels["cell_id"].tolist() if WHICH == "labelled" else None,
                          send_plot=supports_images(PROVIDER),
                          skip_fit=os.path.exists(f"{cfg.out_dir}/summary.csv"))
for g in (agent.get_graph(), build_investigator(llm, []).get_graph()):
    try:
        display(Image(g.draw_mermaid_png()))     # rendered via mermaid.ink (needs internet)
    except Exception:
        print(g.draw_mermaid())

## Run

Fitting all 139 cells takes a few minutes (skipped if fits from an earlier run exist). The investigation makes
roughly 6–8 model calls per cell; at `RPM = 2` that is about 3–4 minutes per cell.

In [ ]:
final = agent.invoke({"queue": [], "verdicts": [], "report": "", "shortlist": []}, {"recursion_limit": 500})
verdicts = pd.DataFrame(final["verdicts"])
verdicts[["cell_id", "category", "confidence", "tool_calls", "revised", "baseline"]]

## Triage: which cells were shortlisted, and why

In [ ]:
triage = pd.read_csv(f"{cfg.out_dir}/summary_triage.csv")
print(f"{int(triage['flag'].sum())} of {len(triage)} cells flagged, "
      f"{int((triage['priority'] == 'high').sum())} high priority")
triage[triage["flag"]][["cell_id", "priority", "best_parametric", "knee_fraction", "fade_completed", "reasons"]]

## Verdicts and evidence

In [ ]:
for _, v in verdicts.iterrows():
    display(Markdown(f"**{v.cell_id}**: `{v.category}` ({v.confidence} confidence), {v.tool_calls} tool calls"
                     + (", revised after the verifier" if v.revised else "") + "\n\n"
                     + "\n".join(f"- {e}" for e in v.evidence) + f"\n\n*Next step:* {v.recommended_action}"))

## Agreement with hand labels

In [ ]:
from battery_fit_agent.evaluate import evaluate
score, table = evaluate(verdicts, "results/manual_labels.csv")
print(score)
table

## Report (written by the model from the verdicts only)

In [ ]:
display(Markdown(final["report"]))

## Every investigation, step by step

Each trace shows the tools the model chose, what they returned, any verifier feedback, and the final verdict.

In [ ]:
for cell in verdicts["cell_id"]:
    path = f"{cfg.out_dir}/traces/{cell}.md"
    if os.path.exists(path):
        display(Markdown(open(path).read()))
        display(Image(f"{cfg.plot_dir}/{cell}.png", width=600))
        display(Markdown("---"))

## Keeping this run

To publish this run with its outputs: in Colab, **File → Save a copy in GitHub** (or **File → Download → .ipynb**
and commit it as `notebooks/colab_investigator.ipynb`). In Jupyter, just save and commit.
The API key is read from a secret and never printed, so it is not stored in the notebook.